<a href="https://colab.research.google.com/github/toddbalwinski/ds2002-fa26/blob/main/labs/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [186]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [187]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [188]:
dupes = df.duplicated().sum()
clean = df.drop_duplicates().copy()
log('duplicates', 'dropped duplicate rows', dupes)


[duplicates] dropped duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [189]:
clean['price'] = (
    clean['price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.strip()
    .astype(float)
)

df['price'] = (
    df['price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.strip()
    .astype(float)
)

print((clean['qty'] * clean['price']).sum())
print((df['qty'] * df['price']).sum())

assert clean['price'].dtype == float

log('price', 'removed dollar signs and whitespace + converted price to float', len(clean))

4594.5
4852.5
[price] removed dollar signs and whitespace + converted price to float (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [190]:
missing_qty = clean['qty'].isna().sum()
negative_qty = (clean['qty'] < 0).sum()

clean = clean[clean['qty'].notna() & (clean['qty'] >= 0)].copy()

log('qty_missing/negative', 'dropped rows with missing/negative quantities', (negative_qty + missing_qty))

[qty_missing/negative] dropped rows with missing/negative quantities (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [191]:
print(clean['item'].value_counts())

before_items = clean['item'].nunique()

ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}

clean['item'] = clean['item'].map(ITEM_MAP)

after_items = clean['item'].nunique()

log('item', f'collapsed item variants from {before_items} to {after_items}', before_items - after_items)

print(clean['item'].value_counts())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] collapsed item variants from 6 to 3 (3 row(s))
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [192]:
print(clean['category'].value_counts())

before_categories = clean['category'].nunique()

CATEGORY_MAP = {
    'Food': 'Food',
    'food': 'Food',
    'Merch': 'Merch',
    'Apparel': 'Merch',
    'RainGear': 'Rain Gear',
    'rain-gear': 'Rain Gear'
}

clean['category'] = clean['category'].map(CATEGORY_MAP)

after_categories = clean['category'].nunique()

log('category', f'normalized categories from {before_categories} to {after_categories}. I combined merch and apparel into merch', before_categories - after_categories)

print(clean['category'].value_counts())

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] normalized categories from 6 to 3. I combined merch and apparel into merch (3 row(s))
category
Food         95
Merch        94
Rain Gear    86
Name: count, dtype: int64


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [193]:
df=clean.copy()
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert set(df['item'].unique()) == {'Cheeseburger', 'Foam Finger', 'Rain Poncho'}
assert set(df['category'].unique()) == {'Food', 'Merch', 'Rain Gear'}
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [194]:
df['revenue'] = df['qty'] * df['price']

revenue_by_category = (
    df.groupby('category')['revenue']
    .sum()
    .sort_values(ascending=False)
    .round(2)
)

print(revenue_by_category)

overall_total = df['revenue'].sum()

print('Overall total:', round(overall_total, 2))

category
Food         1656.0
Merch        1572.0
Rain Gear    1512.0
Name: revenue, dtype: float64
Overall total: 4740.0


**What I would tell the vendor:** _..._

### TODO 8 — read back your log

In [195]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped duplicate rows,15
1,price,removed dollar signs and whitespace + converte...,300
2,qty_missing/negative,dropped rows with missing/negative quantities,25
3,item,collapsed item variants from 6 to 3,3
4,category,normalized categories from 6 to 3. I combined ...,3


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

The step that changed revue the most was dropping duplicates because it was the only one that removed positive revenue rows. The revenue went from 4852.5 to 4594.5 through this step. I think that someone could have reasonably chosen to not combine the merch and apparel categories and instead keep them seperate as there is the potential that the apparel is not merchandise from the game but some other more generic apparel.